In [1]:
import os
import requests
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
# Gemini sometimes returns 503 (overloaded) or 429 (rate limit), so retry a few times.
client = genai.Client(
    http_options=types.HttpOptions(retry_options=types.HttpRetryOptions(attempts=5))
)

OPENWEATHER_URL = "https://api.openweathermap.org/data/2.5/weather"

# The tool saves each temperature here. The average is calculated from
# these values, not taken from the model's reply.
collected_temperatures = {}


def get_current_weather(city: str) -> dict:
    """Gets the current weather for a single city from OpenWeather.

    Args:
        city: The city name, optionally with a country code (e.g., 'Kathmandu' or 'Paris,FR').
    """
    api_key = os.getenv("OPENWEATHER_API_KEY")
    if not api_key:
        return {"error": "OPENWEATHER_API_KEY is not set."}

    try:
        response = requests.get(
            OPENWEATHER_URL,
            params={"q": city, "appid": api_key, "units": "metric"},
            timeout=10,
        )
    except requests.RequestException as e:
        # Don't return str(e). It contains the request URL, and the URL has the API key.
        return {"error": f"Could not reach OpenWeather ({type(e).__name__})."}

    if response.status_code != 200:
        message = response.json().get("message", response.text)
        return {"error": f"OpenWeather error for '{city}': {message}"}

    data = response.json()
    temperature = data["main"]["temp"]
    collected_temperatures[data["name"]] = temperature
    return {
        "city": data["name"],
        "country": data["sys"]["country"],
        "temperature_c": temperature,
        "feels_like_c": data["main"]["feels_like"],
        "description": data["weather"][0]["description"],
    }


available_tools = {"get_current_weather": get_current_weather}

In [2]:
def run_weather_agent(cities: list[str], max_steps: int = 10) -> None:
    """Asks Gemini for the weather in each city, executing tool calls one at a time."""
    collected_temperatures.clear()

    config = types.GenerateContentConfig(
        system_instruction=(
            "You are a weather agent. Call get_current_weather for ONE city at a time, "
            "in the order the user lists them. After all cities are done, report each "
            "city's temperature and the average temperature in °C."
        ),
        tools=[get_current_weather],
        temperature=0.0,
        # Automatic function calling is off. The loop below runs each tool call itself, in order.
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
    )

    prompt = "What is the current temperature in " + ", ".join(cities) + "? Also give the average."
    print(f"User: {prompt}")
    contents = [types.Content(role="user", parts=[types.Part.from_text(text=prompt)])]

    for step in range(1, max_steps + 1):
        response = client.models.generate_content(
            model="gemini-3.5-flash", contents=contents, config=config
        )
        contents.append(response.candidates[0].content)

        if not response.function_calls:
            break  # no tool calls left, so this response is the final answer

        tool_parts = []
        for call in response.function_calls:
            print(f"[STEP {step}] {call.name}({dict(call.args)})")
            tool = available_tools.get(call.name)
            result = tool(**call.args) if tool else {"error": f"Unknown tool {call.name}"}
            print(f"          -> {result}")
            tool_parts.append(types.Part.from_function_response(name=call.name, response=result))
        contents.append(types.Content(role="user", parts=tool_parts))
    else:
        print("Stopped: too many tool steps.")

    print("\n--- Gemini's Answer ---")
    print(response.text)

    print("\n--- Verified by Python ---")
    for city, temp in collected_temperatures.items():
        print(f"{city:<15} {temp:6.2f} °C")
    if len(collected_temperatures) == len(cities):
        average = sum(collected_temperatures.values()) / len(collected_temperatures)
        print(f"{'Average':<15} {average:6.2f} °C")
    else:
        print(f"Only got {len(collected_temperatures)} of {len(cities)} cities, so no average.")


run_weather_agent(["Kathmandu", "London", "Tokyo"])

User: What is the current temperature in Kathmandu, London, Tokyo? Also give the average.


[STEP 1] get_current_weather({'city': 'Kathmandu'})


          -> {'city': 'Kathmandu', 'country': 'NP', 'temperature_c': 19.17, 'feels_like_c': 19.7, 'description': 'overcast clouds'}


[STEP 2] get_current_weather({'city': 'London'})


          -> {'city': 'London', 'country': 'GB', 'temperature_c': 11.8, 'feels_like_c': 11.28, 'description': 'overcast clouds'}


[STEP 3] get_current_weather({'city': 'Tokyo'})


          -> {'city': 'Tokyo', 'country': 'JP', 'temperature_c': 23.62, 'feels_like_c': 23.97, 'description': 'overcast clouds'}



--- Gemini's Answer ---
The current temperatures for the requested cities are:

*   **Kathmandu:** 19.17°C
*   **London:** 11.80°C
*   **Tokyo:** 23.62°C

**Average Temperature:** 18.20°C

--- Verified by Python ---
Kathmandu        19.17 °C
London           11.80 °C
Tokyo            23.62 °C
Average          18.20 °C
